In [2]:
import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path

# ==================== Paths and Parameters ====================
base_dir = Path(r"C:\Users\*****\Desktop\Data_preprocessing")
nc_dir = base_dir / "Mate_Data"
sites_csv = base_dir / "Agricultural_Cultural_Heritage_Site.csv"
output_dir = base_dir

variables = ['CDD', 'R95p', 'RX1day', 'RX5day', 'TX90p', 'TXx']

# ==================== Read site information ====================
sites = pd.read_csv(sites_csv)

columns_to_keep = ['NUM', 'ID', 'Name', 'Cite', 'Lon', 'Lat', 'Type1', 'Type2', 'Type']
existing_columns = [col for col in columns_to_keep if col in sites.columns]
sites = sites[existing_columns]

if 'Lon' in sites.columns and 'Lat' in sites.columns:
    sites = sites.dropna(subset=['Lon', 'Lat']).reset_index(drop=True)
else:
    raise KeyError("Missing Lon or Lat column in CSV file. Please check column names.")

years = list(range(1980, 2026))

# ==================== Batch extraction and export ====================
for var in variables:
    print(f"\n{'='*40}\nStart processing variable: {var}\n{'='*40}")

    nc_file = nc_dir / f"dataset_{var}_1950_2025.nc"
    if not nc_file.exists():
        print(f"File not found: {nc_file}, skipping this variable.")
        continue

    output_csv = output_dir / f"{var}_1980_2025_extracted.csv"

    ds = xr.open_dataset(nc_file)

    if var not in ds.variables:
        print(f"Variable '{var}' not found in file, skipping.")
        ds.close()
        continue

    data_var = ds[var].sel(year=slice(1980, 2025))

    # Get latitude and longitude arrays (for subsequent index lookup)
    lat_arr = data_var.latitude.values
    lon_arr = data_var.longitude.values

    def extract_point(lon, lat, max_radius=5):
        """
        Extract 46-year data for the target latitude and longitude.
        If a year is NaN, fill with the first non-NaN value from
        neighboring grid points (gradually expanding the radius).
        """
        # First, take the nearest grid point
        point = data_var.sel(longitude=lon, latitude=lat, method='nearest')
        values = point.values.squeeze()   # shape (46,)

        # If there are no missing values, return directly
        if not np.any(np.isnan(values)):
            return values

        # ---- Missing values exist, try neighboring filling ----
        # 1. Get the index of the nearest grid point
        nearest_lat = float(point.latitude)
        nearest_lon = float(point.longitude)
        lat_idx = np.abs(lat_arr - nearest_lat).argmin()
        lon_idx = np.abs(lon_arr - nearest_lon).argmin()

        # 2. Get indices of missing years (0-based)
        missing_years_idx = np.where(np.isnan(values))[0]

        # 3. Search neighbors for each missing year
        filled_values = values.copy()
        for yr_idx in missing_years_idx:
            found = False
            for r in range(1, max_radius + 1):
                # Generate offset range
                lat_offsets = range(-r, r + 1)
                lon_offsets = range(-r, r + 1)
                for dlat in lat_offsets:
                    for dlon in lon_offsets:
                        # Only check points on the boundary of radius r (avoid duplicate checking of interior)
                        if max(abs(dlat), abs(dlon)) != r:
                            continue
                        nlat_idx = lat_idx + dlat
                        nlon_idx = lon_idx + dlon
                        # Boundary check
                        if (0 <= nlat_idx < len(lat_arr) and
                            0 <= nlon_idx < len(lon_arr)):
                            neighbor_value = data_var.isel(
                                year=yr_idx,
                                latitude=nlat_idx,
                                longitude=nlon_idx
                            ).values
                            if not np.isnan(neighbor_value):
                                filled_values[yr_idx] = neighbor_value
                                found = True
                                break
                    if found:
                        break
                if found:
                    break
            if not found:
                # If all within radius are NaN, keep NaN (or could print warning)
                print(f"      Warning: Year {years[yr_idx]} ({lon},{lat}) has no valid neighbor within radius {max_radius}")

        return filled_values

    # Iterate over all sites
    results = []
    for _, row in sites.iterrows():
        try:
            values = extract_point(row['Lon'], row['Lat'])

            record = {col: row[col] for col in existing_columns}
            for yr, val in zip(years, values):
                record[str(yr)] = val

            results.append(record)
            print(f"  Completed: {row.get('ID', '')} {row.get('Name', '')}  ({row['Lon']}, {row['Lat']})")
        except Exception as e:
            print(f"  Extraction failed: {row.get('ID', '')} {row.get('Name', '')}  ({row['Lon']}, {row['Lat']}), error: {e}")

    if results:
        result_df = pd.DataFrame(results)
        result_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"Variable {var} extraction completed, results saved to: {output_csv}")
    else:
        print(f"No data extracted for variable {var}.")

    ds.close()

print("\nAll variables processed.")


开始处理变量: CDD
  完成：1 河北宣化传统葡萄园  (115.061, 40.655)
  完成：2 内蒙古敖汉旱作农业系统  (119.916, 42.289)
  完成：3 辽宁鞍山南果梨栽培系统  (123.046, 41.009)
  完成：4 辽宁宽甸柱参传统栽培体系  (125.405, 40.767)
  完成：5 江苏兴化垛田传统农业系统  (119.856, 32.932)
  完成：6 浙江青田稻鱼共生系统  (120.304, 27.995)
  完成：7 浙江绍兴会稽山古香榧群  (120.576, 29.775)
  完成：8 福建福州茉莉花种植与茶文化系统  (119.187, 26.1)
  完成：9 福建尤溪联合梯田  (118.227, 26.373)
  完成：10 江西万年稻作文化系统  (117.132, 28.604)
  完成：11 湖南新化紫鹊界梯田  (110.992, 27.679)
  完成：12 云南红河哈尼稻作梯田系统  (103.373, 23.366)
  完成：13 云南普洱古茶园与茶文化系统  (99.996, 22.218)
  完成：14 贵州从江侗乡稻鱼鸭系统  (108.573, 25.59)
  完成：15 陕西佳县古枣园  (110.494, 38.187)
  完成：16 甘肃皋兰什川古梨园  (104.005, 36.152)
  完成：17 甘肃迭部扎尕那农林牧复合系统  (103.19, 34.239)
  完成：18 新疆吐鲁番坎儿井农业系统  (89.143, 42.944)
  完成：19 天津滨海崔庄古冬枣园  (117.233, 38.629)
  完成：20 河北宽城传统板栗栽培系统  (118.547, 40.342)
  完成：21 河北涉县旱作梯田系统  (113.829, 36.59)
  完成：22 内蒙古阿鲁科尔沁草原游牧系统  (119.69, 44.722)
  完成：23 浙江杭州西湖龙井茶文化系统  (120.102, 30.221)
  完成：24 浙江湖州桑基鱼塘系统  (120.154, 30.757)
  完成：25 浙江庆元香菇文化系统  (119.18, 27.738)
  完成：26 福建安溪铁观音茶文化系统  (117.944